# Experiment 009a: SmolLM2-360M — V2 Generator Validation (1K steps)

End-to-end test of the v2 generator data (formulaic templates, hard negatives,
edge cases) on SmolLM2-360M before scaling to Llama 3.2-1B.

**Baseline:** Exp 004 scored 12.5% at 500 steps and 25.0% at 5K steps on 5.6K data.
If the v2 generator data is better distributed, we should see at least comparable
results at 1K steps.

| Parameter | Value |
|-----------|-------|
| Model | SmolLM2-360M-Instruct |
| Data | ~14K (v2 generator: formulaic + contextual + hard negatives) |
| Steps | 1,000 |
| Effective batch | 16 (4 × 4) |
| Epochs | ~1.2 |
| Distribution | 30% ADD / 30% SUB / 15% BETWEEN / 25% NO_ROUTE |

**Runtime:** T4 is fine. ~15 minutes.

In [ ]:
!pip install -q transformers datasets accelerate torch tensorboard

In [ ]:
import torch, os, subprocess, math, json
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 1. Clone & Generate Data

In [ ]:
if not os.path.exists('tagzeit-gemma-2b'):
    !git clone https://github.com/mgosal/tagzeit-gemma-2b.git
!cd tagzeit-gemma-2b && npm install --silent
print("✅ Ready")

In [ ]:
# Extract hard negatives from ComplexTempQA
!cd tagzeit-gemma-2b && python tools/extract_hard_negatives.py \
    --output data/hard_negatives/complex_tempqa_noroute.jsonl --count 7000

# Generate 14K samples (30/30/15/25 distribution)
# 14K × 0.95 = 13,300 train, 700 eval
# 1K steps × 16 batch = 16K samples seen ≈ 1.2 epochs
!cd tagzeit-gemma-2b && node experiments/temporal-tagzeit/src/generator_route.js \
    --count 14000 \
    --output data/train/train_routed_009a.jsonl \
    --eval data/eval/eval_routed_009a.jsonl \
    --hard-negatives data/hard_negatives/complex_tempqa_noroute.jsonl

In [ ]:
# Format for SmolLM2 chat template
!cd tagzeit-gemma-2b && python tools/format_for_model.py \
    --model_id HuggingFaceTB/SmolLM2-360M-Instruct \
    --input data/train/train_routed_009a.jsonl \
    --output data/train/train_routed_009a_smollm2.jsonl \
    --eval_input data/eval/eval_routed_009a.jsonl \
    --eval_output data/eval/eval_routed_009a_smollm2.jsonl

TRAIN_FILE = 'tagzeit-gemma-2b/data/train/train_routed_009a_smollm2.jsonl'
EVAL_FILE  = 'tagzeit-gemma-2b/data/eval/eval_routed_009a_smollm2.jsonl'

# Spot-check
with open(TRAIN_FILE) as f:
    for i, line in enumerate(f):
        if i >= 2: break
        rec = json.loads(line)
        print(f"--- Sample {i+1} ---")
        print(rec['text'][:300])
        print()

## 2. Load Model + Domain Tokens

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import numpy as np

MODEL_ID = "HuggingFaceTB/SmolLM2-360M-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float32, device_map="auto")

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    model.config.pad_token_id = tokenizer.eos_token_id

# Domain tokens — must match generator_route.js
ROUTE_TOKENS = [
    "[ROUTE]", "[/ROUTE]", "[NO_ROUTE]",
    "[ROUTE_TIME_ADD]", "[ROUTE_TIME_SUB]", "[ROUTE_DURATION_BETWEEN]",
    "[ROUTE_CALENDAR_SHIFT]", "[ROUTE_TIMEZONE_CONVERT]",
    "[HEAD_TIME]", "[HEAD_DURATION]", "[HEAD_DATE]",
    "[REF_1]", "[REF_2]", "[REF_3]",
]
ROUTE_TOKENS += [f"[ARG_HOUR_{str(h).zfill(2)}]" for h in range(24)]
ROUTE_TOKENS += [f"[ARG_MIN_{str(m).zfill(2)}]" for m in range(60)]
ROUTE_TOKENS += ["[ARG_MON]", "[ARG_TUE]", "[ARG_WED]", "[ARG_THU]", "[ARG_FRI]", "[ARG_SAT]", "[ARG_SUN]"]

num_added = tokenizer.add_tokens(ROUTE_TOKENS)
model.resize_token_embeddings(len(tokenizer))

# Geometric sinusoidal init
embed_dim = model.config.hidden_size
with torch.no_grad():
    new_start = len(tokenizer) - num_added
    for i in range(num_added):
        freq = np.array([1.0 / (10000 ** (2 * j / embed_dim)) for j in range(embed_dim)])
        phase = (i + 1) * freq
        init_vec = np.sin(phase) * 0.02
        model.get_input_embeddings().weight[new_start + i] = torch.tensor(init_vec, dtype=torch.float32)

print(f"✅ {MODEL_ID} loaded, {num_added} domain tokens added")
print(f"Params: {sum(p.numel() for p in model.parameters()):,}")

## 3. Tokenize & Train

In [ ]:
from datasets import load_dataset

dataset = load_dataset('json', data_files={'train': TRAIN_FILE, 'eval': EVAL_FILE})

def tokenize_fn(examples):
    result = tokenizer(examples['text'], truncation=True, max_length=256, padding='max_length')
    result['labels'] = result['input_ids'].copy()
    return result

tokenized = dataset.map(tokenize_fn, batched=True, remove_columns=['text'])
tokenized.set_format('torch')

n_train = len(tokenized['train'])
effective_batch = 4 * 4  # per_device × grad_accum
epochs = (1000 * effective_batch) / n_train
print(f"Train: {n_train}, Eval: {len(tokenized['eval'])}")
print(f"1K steps × {effective_batch} batch = {1000 * effective_batch:,} samples seen")
print(f"Epochs: {epochs:.2f}")

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

OUTPUT_DIR = "/content/tagzeit-009a"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    max_steps=1000,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=1e-4,
    lr_scheduler_type="cosine",
    warmup_steps=100,
    weight_decay=0.01,
    optim="adamw_torch",
    fp16=torch.cuda.is_available(),
    bf16=False,
    eval_strategy="steps",
    eval_steps=250,
    save_strategy="steps",
    save_steps=1000,  # Save final checkpoint only
    save_total_limit=1,
    logging_steps=50,
    logging_dir=f"{OUTPUT_DIR}/logs",
    report_to="tensorboard",
    seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized['train'],
    eval_dataset=tokenized['eval'],
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
    processing_class=tokenizer,
)
print(f"✅ Trainer ready — 1K steps, ~{epochs:.1f} epochs, final checkpoint at step 1000")

In [ ]:
train_result = trainer.train()

print(f"\nTrain loss: {train_result.training_loss:.4f}")
print(f"Runtime: {train_result.metrics['train_runtime']:.0f}s")

eval_results = trainer.evaluate()
print(f"Eval loss: {eval_results['eval_loss']:.4f}")
print(f"Eval perplexity: {math.exp(eval_results['eval_loss']):.2f}")

## 4. Inference Test

Test formulaic (harness-matching), contextual, and NO_ROUTE prompts.
Compare to Exp 004 baseline (12.5% at 500 steps).

In [ ]:
test_prompts = [
    # Formulaic — these match the validation harness
    ("What time is it 1 minute after 23:59?", "ADD"),
    ("What time is it 15 minutes after 12:00?", "ADD"),
    ("What time was it 30 minutes before 00:15?", "SUB"),
    ("What time was it 1 hour before 01:00?", "SUB"),
    ("How much time is there between 09:00 and 17:30?", "BETWEEN"),
    # Contextual
    ("The meeting starts at 14:30 and lasts 45 minutes. When does it end?", "ADD"),
    ("I arrived at work at 09:15. The commute was 25 minutes. When did I leave home?", "SUB"),
    # NO_ROUTE
    ("What is 42 + 18?", "NO_ROUTE"),
    ("When was The Secret Garden published?", "NO_ROUTE"),
    ("What timezone is Tokyo in?", "NO_ROUTE"),
]

model.eval()
correct = 0
total = len(test_prompts)

for prompt, expected_type in test_prompts:
    messages = [{"role": "user", "content": prompt}]
    input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=64, do_sample=False)
    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=False)
    response = response.split('<|im_end|>')[0].strip()

    # Quick classification check
    if expected_type == "NO_ROUTE":
        got_right = "[NO_ROUTE]" in response
    elif expected_type == "ADD":
        got_right = "[ROUTE_TIME_ADD]" in response
    elif expected_type == "SUB":
        got_right = "[ROUTE_TIME_SUB]" in response
    elif expected_type == "BETWEEN":
        got_right = "[ROUTE_DURATION_BETWEEN]" in response
    else:
        got_right = False

    if got_right:
        correct += 1

    status = "✅" if got_right else "❌"
    print(f"{status} [{expected_type}] Q: {prompt}")
    print(f"   A: {response}")
    print()

print(f"\n{'='*50}")
print(f"Operation routing accuracy: {correct}/{total} ({correct/total*100:.1f}%)")
print(f"Exp 004 baseline at 500 steps: 12.5%")

In [ ]:
# Save final model for validate_route.py
trainer.save_model(f"{OUTPUT_DIR}/final")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/final")
print(f"✅ Saved to {OUTPUT_DIR}/final")

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {OUTPUT_DIR}/logs